# Assignment 3.2: Decision tree ensembles

All four learning algorithms in this notebook come from libraries: a single decision tree, a
random forest, AdaBoost, and XGBoost. You do not implement any of them. The cross validation search
and the final test are provided as `cv_search` and `test_accuracy`. Your job is to configure each
model, use those two functions to choose its hyperparameters on the training set and test it once on
the test set, and write the report.

Two code cells are yours: `make_model` and `run_analysis`. Everything else is provided and already
runs.

# Import libraries
Do not use any other Python library.

numpy - Linear algebra library for handling vectors and matrices, collectively processed as numpy arrays.

matplotlib - Graphing library for visualizing results.

csv - Library used for reading csv files into python lists.

sklearn - Machine learning library from which we source the digits dataset, the train and test split, cross validation, and the decision tree, random forest, and AdaBoost classifiers.

xgboost - Library that provides the XGBoost classifier.

time - Simple library for timing code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from csv import reader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from time import time

# Load Datasets

These functions load the two datasets.

`load_digits_data` returns the 1797 handwritten digits, each one 64 pixels with an intensity from
0 to 16. There are ten classes.

`load_credit` returns the 1000 loan applications of the Statlog German Credit dataset from
`../data/credit.csv`. Each application has 20 numeric features, and the target is 1 for a
creditworthy applicant and 0 otherwise.

Outputs:

*   **features**: numpy array of shape (N, number of features)
*   **labels**: numpy array of N integer class labels

# Function: split_dataset

Splits one dataset into a training set and a test set. 30% of the instances go to the test set.
The split is stratified, so both sets keep the class proportions of the full dataset, and it is
seeded, so every run gets the same split. All cross validation happens inside the training set.
The test set is used once per model, at the very end.

Inputs:
*   **features**: numpy array of shape (N, number of features)
*   **labels**: numpy array of N class labels

Outputs:
*   **X_train**, **y_train**: training features and labels
*   **X_test**, **y_test**: test features and labels

In [ ]:
def load_digits_data(plot=True):
    # optical digits dataset from scikit-learn
    digits = load_digits()
    features = digits.data
    labels = digits.target

    if plot:
        images = features.reshape(-1, 8, 8)
        plt.figure(figsize=(10, 5))
        for i in range(10):
            plt.subplot(3, 4, i + 1)
            plt.imshow(images[i], cmap='gray')
            plt.title(f'Handwritten {labels[i]}')
            plt.axis('off')
        plt.tight_layout()
        plt.show()

    return features, labels


def load_credit(plot=True):
    # statlog german credit data, already numeric, last column is the class
    with open('../data/credit.csv', 'r') as file:
        rows = list(reader(file))[1:]
    features = np.array([[float(value) for value in row[:-1]] for row in rows])
    labels = np.array([int(row[-1]) for row in rows])

    if plot:
        plt.figure(figsize=(5, 4))
        plt.bar(['0', '1'], [np.sum(labels == 0), np.sum(labels == 1)])
        plt.xlabel('Class')
        plt.ylabel('applications')
        plt.title('German Credit class counts')
        plt.show()

    return features, labels


def split_dataset(features, labels):
    return train_test_split(features, labels, test_size=0.3, random_state=0, stratify=labels)

# Hyperparameter grids

The values the spec asks you to search. `None` as a tree depth means the tree has no depth limit
and keeps splitting until every leaf is pure. Every search uses `K_FOLDS` folds of the training set.

In [ ]:
K_FOLDS = 5
ENSEMBLES = ["random forest", "adaboost", "xgboost"]

# step 1: single tree depths
TREE_DEPTHS = [1, 5, None]
# step 3: ensemble tree depths, searched with a fixed tree count
ENSEMBLE_DEPTHS = [1, 2, 3]
DEPTH_SEARCH_COUNT = 100
# step 4: ensemble tree counts, searched with a fixed tree depth
TREE_COUNTS = [5, 10, 50, 100]
COUNT_SEARCH_DEPTH = 3

# Function: make_model

Builds one unfitted classifier. The four algorithms come from three different library classes plus
XGBoost, and each one names its hyperparameters in its own way. Look up the constructor of each class
to find which argument sets the tree depth, which sets the number of trees, and which sets how many
features a split may consider.

*   **decision tree**: one scikit-learn decision tree that splits on the Gini index. It has no tree count, so `n_estimators` is ignored.
*   **random forest**: a bagged forest in which each split considers a random sample of features whose size is the square root of the total number of features.
*   **adaboost**: scikit-learn AdaBoost whose base learner is a decision tree with the given depth.
*   **xgboost**: an `XGBClassifier` with the given depth and tree count.

Seed every model, and every tree inside an ensemble, with `random_state=0` so that every run gives
the same numbers. Where a class can train on several CPU cores at once, let it use all of them.

Inputs:
*   **name**: one of "decision tree", "random forest", "adaboost", "xgboost" (string)
*   **max_depth**: maximum depth of each tree, or None for no limit
*   **n_estimators**: number of trees in the ensemble (integer), or None for the single tree

Output:
*   **model**: an unfitted classifier with `fit`, `predict`, and `score` methods

In [ ]:
def make_model(name, max_depth=None, n_estimators=None):
    # 1. single tree: gini splits, the given depth, seeded
    # 2. random forest: the given depth and tree count, square root feature sampling at each split, seeded
    # 3. adaboost: a seeded tree of the given depth as the base learner, the given tree count, seeded
    # 4. xgboost: the given depth and tree count, seeded
    # TODO
    pass

# Function: cv_search

This function is provided. It searches one hyperparameter of one algorithm by K-fold cross
validation on the training set. For each candidate value it builds a fresh model with your
`make_model`, scores it on the `K_FOLDS` folds, and keeps the mean accuracy over the folds. The
other hyperparameter stays at the value given in `fixed`. The best value is the one with the highest
mean accuracy; on a tie, it takes the one that comes first in `values`. It uses scikit-learn's
default folds for a classifier, so every algorithm is scored on exactly the same folds.

Inputs:
*   **name**: algorithm name, as in `make_model`
*   **X_train**, **y_train**: training features and labels
*   **param_name**: the hyperparameter being searched, "max_depth" or "n_estimators" (string)
*   **values**: list of candidate values for that hyperparameter
*   **fixed**: dictionary holding the other hyperparameter and its value, e.g. {"n_estimators": 100}, or an empty dictionary for the single tree

Outputs:
*   **mean_accuracies**: list of mean cross validation accuracies as percentages, one per value (floats)
*   **best_value**: the value with the highest mean accuracy

In [ ]:
def cv_search(name, X_train, y_train, param_name, values, fixed):
    # one fresh model per candidate value, all scored on the same folds
    mean_accuracies = []
    for value in values:
        settings = dict(fixed)
        settings[param_name] = value
        model = make_model(name, **settings)
        fold_scores = cross_val_score(model, X_train, y_train, cv=K_FOLDS)
        mean_accuracies.append(float(np.mean(fold_scores)) * 100)
    best_value = values[int(np.argmax(mean_accuracies))]
    return mean_accuracies, best_value

# Function: test_accuracy

This function is provided. It trains one model from your `make_model` on the full training set and
scores it once on the test set. This is steps 2 and 5 of the spec.

Inputs:
*   **name**: algorithm name, as in `make_model`
*   **X_train**, **y_train**: training features and labels
*   **X_test**, **y_test**: test features and labels
*   **max_depth**: tree depth to use
*   **n_estimators**: tree count to use, or None for the single tree

Output:
*   **accuracy**: test accuracy as a percentage (float)

In [ ]:
def test_accuracy(name, X_train, y_train, X_test, y_test, max_depth, n_estimators):
    # train on the full training set, score once on the test set
    model = make_model(name, max_depth=max_depth, n_estimators=n_estimators)
    model.fit(X_train, y_train)
    return float(model.score(X_test, y_test)) * 100

# Function: run_analysis

Runs steps 1 to 5 of the spec on one dataset and collects the results. Each step is a call to the
provided `cv_search` or `test_accuracy`; you choose the model name, the hyperparameter, the grid, and
the fixed value for each call, and carry the values each search chooses into the tests.

1.  Single tree: search the depth over `TREE_DEPTHS`.
2.  Single tree: test it at its best depth.
3.  Each ensemble in `ENSEMBLES`: search the depth over `ENSEMBLE_DEPTHS` with `DEPTH_SEARCH_COUNT` trees.
4.  Each ensemble: search the tree count over `TREE_COUNTS` with depth `COUNT_SEARCH_DEPTH`.
5.  Each ensemble: test it with its best depth from step 3 and its best tree count from step 4.

The cells below print and plot whatever this function returns, so the dictionary has to have
exactly this shape. It maps each algorithm name, "decision tree" first and then the names in
`ENSEMBLES`, to a dictionary with these five keys:

*   **depth_scores**: list of mean cross validation accuracies, one per depth searched
*   **best_depth**: the depth that search chose
*   **count_scores**: list of mean cross validation accuracies, one per tree count searched, or None for the single tree
*   **best_count**: the tree count that search chose, or None for the single tree
*   **test_accuracy**: test accuracy of the final model (float)

Inputs:
*   **X_train**, **y_train**: training features and labels
*   **X_test**, **y_test**: test features and labels

Output:
*   **results**: the dictionary described above

In [ ]:
def run_analysis(X_train, y_train, X_test, y_test):
    # 1. call cv_search on the single tree's depth grid, then call test_accuracy at the depth it chose
    # 2. for each ensemble, call cv_search on its depth grid with the tree count held fixed
    # 3. for each ensemble, call cv_search on its tree count grid with the depth held fixed
    # 4. call test_accuracy for each ensemble with the depth and the tree count its two searches chose
    # 5. store the scores, the chosen values, and the test accuracy under the five keys above
    # TODO
    pass

# Function: print_search_tables

Prints the mean cross validation accuracy of every value searched, marks the chosen value with a
star, and prints the test accuracy of each final model with the settings it used.

# Function: plot_searches

Draws three panels for one dataset: the single tree depth search, the ensemble depth search, and the
ensemble tree count search. The y axis is the mean cross validation accuracy.

Inputs:
*   **results**: the dictionary returned by `run_analysis`
*   **dataset_name**: name of the dataset, used in the titles (string)

In [ ]:
def print_search_tables(results, dataset_name):
    print(f"{dataset_name}: mean {K_FOLDS} fold cross validation accuracy, * marks the chosen value")
    tree = results["decision tree"]
    print("  decision tree, max_depth")
    for depth, score in zip(TREE_DEPTHS, tree["depth_scores"]):
        mark = "*" if depth == tree["best_depth"] else " "
        print(f"    {str(depth):>5s} {score:6.2f}% {mark}")

    for name in ENSEMBLES:
        result = results[name]
        print(f"  {name}, max_depth with {DEPTH_SEARCH_COUNT} trees")
        for depth, score in zip(ENSEMBLE_DEPTHS, result["depth_scores"]):
            mark = "*" if depth == result["best_depth"] else " "
            print(f"    {str(depth):>5s} {score:6.2f}% {mark}")
        print(f"  {name}, n_estimators with max_depth {COUNT_SEARCH_DEPTH}")
        for count, score in zip(TREE_COUNTS, result["count_scores"]):
            mark = "*" if count == result["best_count"] else " "
            print(f"    {str(count):>5s} {score:6.2f}% {mark}")

    print(f"{dataset_name}: test accuracy of the final models")
    for name, result in results.items():
        print(f"  {name:14s} max_depth={str(result['best_depth']):5s} "
              f"n_estimators={str(result['best_count']):5s} {result['test_accuracy']:6.2f}%")


def plot_searches(results, dataset_name):
    figure, axes = plt.subplots(1, 3, figsize=(15, 4))

    tree = results["decision tree"]
    axes[0].bar([str(depth) for depth in TREE_DEPTHS], tree["depth_scores"])
    axes[0].set_xlabel("max_depth")
    axes[0].set_title(f"{dataset_name}, decision tree depth")

    for name in ENSEMBLES:
        axes[1].plot(ENSEMBLE_DEPTHS, results[name]["depth_scores"], marker="o", label=name)
        axes[2].plot(TREE_COUNTS, results[name]["count_scores"], marker="o", label=name)
    axes[1].set_xticks(ENSEMBLE_DEPTHS)
    axes[1].set_xlabel("max_depth")
    axes[1].set_title(f"{dataset_name}, ensemble depth ({DEPTH_SEARCH_COUNT} trees)")
    axes[2].set_xscale("log")
    axes[2].set_xticks(TREE_COUNTS, [str(count) for count in TREE_COUNTS])
    axes[2].set_xlabel("n_estimators")
    axes[2].set_title(f"{dataset_name}, ensemble tree count (max_depth {COUNT_SEARCH_DEPTH})")

    for axis in axes:
        axis.set_ylabel("mean CV accuracy (%)")
    axes[1].legend()
    axes[2].legend()
    plt.tight_layout()
    plt.show()

# Main code

Loads each dataset, splits it, runs your analysis, then prints the search tables and draws the
search plots. For German Credit it also prints the share of the majority class in the test set,
which is the accuracy of a model that always predicts that class. The whole cell takes a few
minutes; most of the time goes to AdaBoost and the random forest on the digits.

In [ ]:
datasets = {
    "digits": load_digits_data,
    "credit": load_credit,
}

all_results = {}
total_start = time()

for dataset_name, load_dataset in datasets.items():
    features, labels = load_dataset()
    X_train, X_test, y_train, y_test = split_dataset(features, labels)
    print(f"{dataset_name}: {len(y_train)} training and {len(y_test)} test instances, "
          f"{features.shape[1]} features, {len(np.unique(labels))} classes")
    majority = np.max(np.bincount(y_test)) / len(y_test) * 100
    print(f"majority class share of the test set: {majority:.2f}%")

    start_time = time()
    results = run_analysis(X_train, y_train, X_test, y_test)
    print(f"elapsed time: {time() - start_time:.1f}s")

    all_results[dataset_name] = results
    print_search_tables(results, dataset_name)
    plot_searches(results, dataset_name)

print(f"total time: {time() - total_start:.1f}s")

# Comparison table

Test accuracy of all eight final models, one row per algorithm and one column per dataset, with the
depth and tree count each one was trained with.

In [ ]:
print(f"{'':14s}" + "".join(f"  {name:<22s}" for name in all_results))
print(f"{'algorithm':14s}" + "  accuracy  depth  trees" * len(all_results))
for name in ["decision tree"] + ENSEMBLES:
    row = f"{name:14s}"
    for results in all_results.values():
        result = results[name]
        row += (f"  {result['test_accuracy']:7.2f}%  {str(result['best_depth']):>5s}"
                f"  {str(result['best_count']):>5s}")
    print(row)

# Report

Cover the following for this notebook.

*   A table for each of the fourteen hyperparameter searches: the single tree depth on each dataset, and the depth and the tree count of each ensemble on each dataset. Give the mean cross validation accuracy of every value and mark the value chosen.
*   The search plots for each dataset.
*   The comparison table of the eight test accuracies, with the depth and tree count each final model used.
*   How tree depth affects each ensemble. Compare the random forest with the two boosting methods, and explain the difference in terms of how bagging and boosting combine their trees.
*   How the tree count affects each ensemble, and where adding trees stops helping.
*   The single tree against the ensembles on each dataset. Say which dataset gains more from an ensemble and why.
*   German Credit test accuracy against the majority class share printed above. Say what that baseline tells you about how hard this dataset is.
*   Cross validation accuracy against test accuracy for the chosen models. Say whether cross validation picked well, and where the two disagree.
*   Why the test set is used only once per model, after cross validation has chosen the hyperparameters.